# Детекция заправок по временным рядам

Находить события заправки по телеметрии датчика уровня топлива, а не только классифицировать отдельные временные точки.

[Описание и данные](README.md) · [История эксперимента](results/recorded_metrics.json)

**Запуск:** Python 3.11 или 3.12. Полное обучение требует данных и подходящего GPU. Выводы очищены; численные результаты ниже появятся после выполнения.

## Подготовка окружения

Ячейка устанавливает зависимости проекта. Локально рекомендуется отдельное виртуальное окружение.

In [ ]:
%pip -q install -r https://raw.githubusercontent.com/Xokage9/ml-portfolio/main/projects/02_fuel_events/requirements.txt


In [ ]:
import os
os.environ['OMP_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'
from threadpoolctl import threadpool_limits
threadpool_limits(limits=2)


## 1. Данные

Файл `fuel_5min.csv` — реальные данные автопарка: 50 машин, полгода работы
(июль–декабрь 2025), 6.1 млн исходных сообщений, свёрнутых в **5-минутные интервалы**.
Номера машин обезличены, координаты убраны.

| колонка | что это |
|---|---|
| `vehicle` | идентификатор ТС, `V01`…`V50` |
| `ts` | начало 5-минутного интервала |
| `fuel` | средний уровень топлива в литрах по валидным сообщениям интервала |
| `ign` | доля времени с включённым зажиганием, 0..1 |
| `speed` | средняя скорость |
| `n_msg` | сколько исходных сообщений попало в интервал |
| `label` | **0** — ничего, **1** — заправка с включённым двигателем, **2** — заправка с выключенным двигателем |
| `split` | `train` / `test`, разбиение по машинам |

**Про классы.** Наша цель — класс **1**. Класс 2 (заправка с выключенным двигателем) в этих данных
выглядит принципиально иначе: датчик молчал всю заправку, и в 5-минутном ряду от неё
остаётся только скачок уровня между двумя соседними интервалами. Мы его не выкидываем,
но и не считаем целевым — модель учится отличать все три состояния.

**Про разбиение.** `split` делит выборку **по машинам, а не по точкам**.
Если резать случайно по точкам, окна одного и того же события попадут и в train, и в test,
и метрика будет завышена. Не меняйте это разбиение.

Из обучающих машин мы дополнительно откладываем девять под валидацию: на ней
выбирается лучшая эпоха. Тест используется ровно один раз — для итогового замера.
Выбирать эпоху по тесту нельзя: результат окажется завышенным, а на новых машинах
модель сработает хуже, чем обещала метрика.


In [ ]:
import os
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Откуда брать данные. Если ноутбук открыт из GitHub, ссылка уже прописана
# и ничего делать не нужно — файл скачается сам.
DATA_URL = "https://raw.githubusercontent.com/hehehelp123/fuel-refuel-lab/main/data/fuel_5min.csv"
LOCAL = "fuel_5min.csv"

if not os.path.exists(LOCAL):
    if DATA_URL and not DATA_URL.startswith("__"):
        print("качаю датасет...")
        urllib.request.urlretrieve(DATA_URL, LOCAL)
    else:
        # Запасной вариант: загрузить файл кнопкой вручную
        from google.colab import files
        uploaded = files.upload()
        LOCAL = list(uploaded)[0]

df = pd.read_csv(LOCAL, parse_dates=["ts"])
print("строк:", len(df), "| машин:", df.vehicle.nunique())
df.head()


In [ ]:
from pathlib import Path
import pickle
USE_TRAINING_CACHE=False
CACHED={}
checkpoint=Path('fuel_artifacts/training_checkpoint.pkl')
if USE_TRAINING_CACHE and checkpoint.exists():
    with open(checkpoint,'rb') as f:CACHED=pickle.load(f)
    pd.testing.assert_frame_equal(df,CACHED['raw'],check_dtype=False)
    print('TRAINING_CACHE: данные совпали; повторная оценка обучения текущей сессии Colab')
else:
    print('FRESH_TRAINING: обучение моделей с нуля')


In [ ]:
def count_events(labels, positive=1):
    """Число непрерывных блоков нужного класса."""
    arr = np.asarray(labels) == positive
    padded = np.pad(arr, (1, 1), constant_values=False)
    return len(np.where(np.diff(padded))[0]) // 2


print("Распределение меток по интервалам:")
print(df.label.value_counts().sort_index().rename({0: "0 — ничего",
                                                   1: "1 — двигатель включён",
                                                   2: "2 — двигатель выключен"}))
print()
share = (df.label == 1).mean() * 100
print(f"Доля целевого класса: {share:.2f}%")
print()

for name, part in df.groupby("split"):
    ev = sum(count_events(g.label) for _, g in part.groupby("vehicle"))
    print(f"{name:5s}: машин {part.vehicle.nunique():2d}, интервалов {len(part):6d}, событий класса 1: {ev}")


### 1.1. Смотрим на заправку глазами

Прежде чем обучать что-либо, полезно посмотреть, как событие выглядит в данных.
Найдём машину с самым большим числом событий и нарисуем одно из них.


In [ ]:
def events_of(g, positive=1):
    """Границы блоков класса positive внутри одной машины: список (i_start, i_end)."""
    lab = (g.label.values == positive)
    out, start = [], None
    for i, v in enumerate(lab):
        if v and start is None:
            start = i
        elif not v and start is not None:
            out.append((start, i - 1))
            start = None
    if start is not None:
        out.append((start, len(lab) - 1))
    return out


top = (df[df.label == 1].groupby("vehicle").size().sort_values(ascending=False).index[0])
g = df[df.vehicle == top].reset_index(drop=True)
evts = events_of(g)
print(f"машина {top}: событий {len(evts)}")

s, e = evts[len(evts) // 2]                 # берём событие из середины списка
lo, hi = max(0, s - 12), min(len(g) - 1, e + 12)
w = g.iloc[lo:hi + 1]

plt.figure(figsize=(9, 3))
plt.plot(w.ts, w.fuel, marker="o", ms=3, lw=1.2, color="#1a1a1a", label="уровень топлива")
plt.axvspan(g.ts[s], g.ts[e], color="#c8c8c8", alpha=0.6, label="размеченная заправка")
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.ylabel("литры")
plt.xlabel(f"{top}, {g.ts[s]:%d.%m.%Y}")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(w[["ts", "fuel", "ign", "speed", "n_msg", "label"]].to_string(index=False))


Заправка выглядит как монотонный рост уровня на протяжении нескольких интервалов.
Модель должна научиться отличать такой рост от колебаний уровня и шума датчика.

Нули в столбце `fuel` — это интервалы, в которых датчик не передавал показания
(двигатель выключен). Это не пропуски данных, а информативный признак.


## 2. Признаки и окна

Признаки на один интервал — намеренно простые:

* `fuel` — сам уровень;
* `d1..d4` — разности с уровнем на 1, 2, 3 и 4 интервала назад.

Зачем четыре разности разной глубины: `d1` ловит резкий скачок между соседними интервалами,
`d4` — медленный рост, размазанный на 20 минут. Так сеть видит и то, и другое
без явного сглаживающего фильтра и без порогов, подобранных руками.

Дальше нарезаем **скользящие окна по 7 интервалов** (35 минут). Класс окна — это класс
его **центрального** интервала. Центр, а не конец, потому что для решения «была ли тут заправка»
одинаково полезен контекст и до, и после момента.


In [ ]:
FEATURES = ["fuel", "d1", "d2", "d3", "d4"]
SEQ_LEN = 7          # длина окна в интервалах
CENTER = SEQ_LEN // 2


def add_features(g):
    g = g.sort_values("ts").copy()
    for k in (1, 2, 3, 4):
        g[f"d{k}"] = g["fuel"].diff(k).fillna(0.0)
    return g


df = df.groupby("vehicle", group_keys=False).apply(add_features)
df[["vehicle", "ts", "fuel"] + [f"d{k}" for k in (1, 2, 3, 4)]].head(8)


In [ ]:
def make_windows(frame):
    """Нарезает окна внутри каждой машины отдельно, чтобы не склеить разные ТС."""
    X, y, veh, ts = [], [], [], []
    for v, g in frame.groupby("vehicle"):
        vals = g[FEATURES].values.astype("float32")
        lab = g["label"].values.astype("int64")
        times = g["ts"].values
        for i in range(len(g) - SEQ_LEN + 1):
            X.append(vals[i:i + SEQ_LEN])
            y.append(lab[i + CENTER])
            veh.append(v)
            ts.append(times[i + CENTER])
    meta = pd.DataFrame({"vehicle": veh, "ts": pd.to_datetime(ts)})
    return np.asarray(X), np.asarray(y), meta


t0 = time.time()
# Из обучающих машин откладываем часть под валидацию: на ней выбираем эпоху.
# Тест не участвует в выборе эпохи. Если выбирать эпоху по тесту,
# результат будет завышен, и на новых машинах модель окажется хуже.
train_veh = sorted(df[df.split == "train"].vehicle.unique())
ev_per_veh = {v: count_events(df[df.vehicle == v].label) for v in train_veh}
VAL_VEH = set(sorted(train_veh, key=lambda v: -ev_per_veh[v])[1::5][:9])

X_train, y_train, meta_train = make_windows(
    df[df.split.eq("train") & ~df.vehicle.isin(VAL_VEH)])
X_val, y_val, meta_val = make_windows(df[df.vehicle.isin(VAL_VEH)])
X_test, y_test, meta_test = make_windows(df[df.split == "test"])
print(f"нарезка заняла {time.time() - t0:.1f} c")
print("train:", X_train.shape, "| валидация:", X_val.shape, "| test:", X_test.shape)
print("классы в train:", np.bincount(y_train))
print("классы в val  :", np.bincount(y_val))
print("классы в test :", np.bincount(y_test))


## 3. Модель

Двухслойная LSTM на 64 скрытых нейрона, dropout 0.2 между слоями, сверху линейный слой
на 3 класса. Берём скрытое состояние последнего элемента окна.

Это ровно та архитектура, что крутится в проде, — чтобы обученную здесь модель можно
было забрать в рабочий сервис без переписывания.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("устройство:", DEVICE)


class FuelLSTM(nn.Module):
    def __init__(self, n_features=5, hidden=64, layers=2, n_classes=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, layers,
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


model = FuelLSTM(n_features=len(FEATURES)).to(DEVICE)
print(model)
print("параметров:", sum(p.numel() for p in model.parameters()))


### 3.1. Дисбаланс классов

Целевой класс занимает меньше 2% интервалов. Если ничего не делать, модель выучит
«всегда 0» и получит 98% accuracy, не найдя ни одной заправки.

Стандартный приём — веса классов в функции потерь. Но брать вес обратно
пропорционально частоте (`N_max / N_c`) здесь слишком грубо: вес улетает в десятки,
и модель начинает видеть заправки повсюду. Берём **корень** из этого отношения —
градиент от редкого класса становится заметным, но не доминирует.


In [ ]:
def class_weights(y, n_classes=3, max_weight=30.0):
    counts = np.bincount(y, minlength=n_classes)
    majority = counts.max()
    w = [min(np.sqrt(majority / c), max_weight) if c > 0 else 0.0 for c in counts]
    return torch.tensor(w, dtype=torch.float32)


weights = class_weights(y_train)
for i, (c, wt) in enumerate(zip(np.bincount(y_train), weights)):
    print(f"класс {i}: {c:6d} примеров, вес {wt:.2f}")


## 4. Обучение

In [ ]:
EPOCHS = 30
BATCH = 512
LR = 1e-3

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_ds = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False)

criterion = nn.CrossEntropyLoss(weight=weights.to(DEVICE))
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max",
                                                       factor=0.5, patience=2)

from sklearn.metrics import f1_score


def predict(loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            out = model(xb.to(DEVICE))
            preds.append(out.argmax(1).cpu().numpy())
    return np.concatenate(preds)


if CACHED:
    loaded=torch.load('fuel_artifacts/baseline_weights.pt',map_location=DEVICE,weights_only=True)
    assert loaded['features']==FEATURES and loaded['window']==7
    model.load_state_dict(loaded['state_dict'])
    history=CACHED['history']
    assert len(history)==EPOCHS
    best_f1=max(row[2] for row in history)
    print('BASELINE_WEIGHTS_REUSED: сохранённые веса сегодняшнего обучения',best_f1)
else:
    best_f1, best_state = -1.0, None
    history = []                       # (эпоха, loss, F1) — пригодится для графика
    t0 = time.time()
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item()
    
        val_pred = predict(val_loader)
        f1_target = f1_score(y_val, val_pred, labels=[1], average="macro", zero_division=0)
        scheduler.step(f1_target)
        history.append((epoch, total / len(train_loader), f1_target))
    
        if f1_target > best_f1:
            best_f1 = f1_target
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            mark = "  <- лучшая"
        else:
            mark = ""
    
        print(f"эпоха {epoch:2d} | loss {total / len(train_loader):.4f} | "
              f"F1 на валидации: {f1_target:.3f}{mark}")
    
    model.load_state_dict(best_state)
    print(f"\nобучение заняло {time.time() - t0:.0f} c, лучший F1 на валидации: {best_f1:.3f}")


## 5. Метрики

### 5.1. Точечные метрики


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = predict(test_loader)

names = ["0 — ничего", "1 — двигатель включён", "2 — двигатель выключен"]
print(classification_report(y_test, y_pred, target_names=names,
                            digits=3, zero_division=0))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.imshow(cm / cm.sum(axis=1, keepdims=True).clip(min=1), cmap="Greys", vmin=0, vmax=1.3)
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                color="white" if cm[i, j] / max(cm[i].sum(), 1) > 0.55 else "black")
ax.set_xticks(range(3), ["0", "1", "2"])
ax.set_yticks(range(3), ["0", "1", "2"])
ax.set_xlabel("предсказано")
ax.set_ylabel("на самом деле")
ax.grid(False)
plt.tight_layout()
plt.show()


Класс 2 в тесте представлен всего парой десятков интервалов, поэтому его строчка
в отчёте шумная — не делайте по ней выводов. Целевой класс — первый.

### 5.2. Почему точечных метрик мало

Оператору автопарка не нужен ответ «этот пятиминутный интервал относится к заправке».
Ему нужно «14 августа в 10:05 машина V11 заправилась». Это **событие**, а не точка.

Разница существенная. Модель, которая на каждой заправке угадала 3 интервала из 5,
по точечной метрике выглядит средне, а по событийной — идеально: событие найдено.
И наоборот, модель, которая ровно один интервал посреди пустой стоянки пометила заправкой,
по точечной метрике почти не пострадает, а по событийной получит ложное срабатывание.

Считаем так: собираем предсказанные и эталонные события как непрерывные блоки класса 1
и сопоставляем их с допуском **±5 минут**. Предсказание, попавшее на класс 2,
не штрафуем — там разметке доверять нельзя.


In [ ]:
TOLERANCE = np.timedelta64(5, "m")


def blocks(mask, times):
    """Непрерывные блоки True -> список (t_start, t_end)."""
    out, start = [], None
    for i, v in enumerate(mask):
        if v and start is None:
            start = i
        elif not v and start is not None:
            out.append((times[start], times[i - 1]))
            start = None
    if start is not None:
        out.append((times[start], times[len(mask) - 1]))
    return out


def event_metrics(y_true, y_pred, meta, verbose=True):
    n_true = n_true_hit = 0        # эталонные события и сколько из них найдено
    n_pred = n_pred_hit = 0        # предсказанные события и сколько попало в цель

    for v, sub in meta.groupby("vehicle"):
        pos = sub.index.to_numpy()          # meta идёт в том же порядке, что y_true/y_pred
        times = sub["ts"].values
        t_true = blocks(y_true[pos] == 1, times)
        t_pred = blocks(y_pred[pos] == 1, times)
        t_amb = blocks(y_true[pos] == 2, times)

        hit_true = set()
        for ps, pe in t_pred:
            hit = [k for k, (ts_, te_) in enumerate(t_true)
                   if ps <= te_ + TOLERANCE and pe >= ts_ - TOLERANCE]
            if hit:
                hit_true.update(hit)
                n_pred += 1
                n_pred_hit += 1
            elif any(ps <= te_ + TOLERANCE and pe >= ts_ - TOLERANCE
                     for ts_, te_ in t_amb):
                pass                      # попали в неоднозначную зону — не штрафуем
            else:
                n_pred += 1               # ложное срабатывание
        n_true += len(t_true)
        n_true_hit += len(hit_true)

    # событие засчитывается один раз, сколько бы кусков модель на него ни выдала
    tp, fn = n_true_hit, n_true - n_true_hit
    fp = n_pred - n_pred_hit
    precision = n_pred_hit / n_pred if n_pred else 0.0
    recall = n_true_hit / n_true if n_true else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    if verbose:
        print(f"события: TP={tp}  FP={fp}  FN={fn}")
        print(f"precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")
    return {"tp": tp, "fp": fp, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}


lstm_events = event_metrics(y_test, y_pred, meta_test)


## 6. С чем сравнивать: пороговое правило

Способ, который применяют по умолчанию, — порог на приросте уровня: если уровень
вырос больше чем на X литров за интервал, считаем это заправкой. Посмотрим, какой
результат он даёт на тех же данных. Это нижняя граница, с которой сравнивают модель.


In [ ]:
d1_val=X_val[:,CENTER,FEATURES.index('d1')]
thr_results=[]
for threshold in [1,2,3,5,8,12,20]:
    metrics=event_metrics(y_val,(d1_val>threshold).astype('int64'),meta_val,verbose=False)
    thr_results.append({'threshold_l':threshold,**metrics})
threshold_validation=pd.DataFrame(thr_results)
best_thr=int(threshold_validation.loc[threshold_validation.f1.idxmax(),'threshold_l'])
baseline_rule_threshold=best_thr
print('Порог выбран только на validation:',best_thr)
display(threshold_validation)
d1_center=X_test[:,CENTER,FEATURES.index('d1')]
frozen_threshold_test=event_metrics(y_test,(d1_center>best_thr).astype('int64'),meta_test,verbose=False)
print('Тест при зафиксированном пороге:',frozen_threshold_test)
print('LSTM baseline test event F1:',lstm_events['f1'])


## 7. Итог бейзлайна

Ячейка ниже собирает метрики бейзлайна в один отчёт.


In [ ]:
from sklearn.metrics import precision_score, recall_score

def report(y_true, y_pred, meta, title="модель"):
    p = precision_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    r = recall_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    f = f1_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    ev = event_metrics(y_true, y_pred, meta, verbose=False)
    print(f"=== {title} ===")
    print(f"точечные (класс 1): precision={p:.3f}  recall={r:.3f}  F1={f:.3f}")
    print(f"macro F1 по трём классам: {macro:.3f}")
    print(f"событийные: TP={ev['tp']} FP={ev['fp']} FN={ev['fn']}  "
          f"precision={ev['precision']:.3f} recall={ev['recall']:.3f} F1={ev['f1']:.3f}")
    return {"point_f1": f, "event_f1": ev["f1"]}


baseline = report(y_test, y_pred, meta_test, "LSTM baseline")


## 8. Задание

**Цель: увеличить событийный F1 на тестовых машинах.** Бейзлайн — 0.786, планка «хорошо» — выше 0.90.

Главная метрика — `event_f1` из отчёта выше. Вторая метрика — точечный F1 по классу 1;
её тоже указывайте, чтобы было видно, за счёт чего вырос результат.

### Правила

1. Разбиение `split` не трогаем. Обучаться на машинах из `test` нельзя.
2. Метки в тесте не меняем.
3. Гиперпараметры подбираем на train (можно отрезать часть train-машин под валидацию).
   Если крутить их прямо по тесту, вы просто переобучитесь на пять машин.
4. В отчёте — таблица «что поменял → какой стал event_f1», а не только финальное число.

### Что можно попробовать

От простого к сложному:

* **Признаки.** Сейчас используется только уровень и четыре разности. В данных есть
  `ign`, `speed`, `n_msg`, время суток. Заправка с включённым двигателем — это почти всегда малая скорость
  при включённом зажигании. Помогает ли это?
* **Нормализация.** Признаки поданы в сыром виде: уровень в литрах (0–100), разности около нуля.
  Попробуйте стандартизацию — статистики считайте **только по train**.
* **Длина окна.** 7 интервалов = 35 минут. Длиннее — больше контекста и меньше окон; короче — наоборот.
* **Веса классов.** Мы взяли корень из отношения частот. Попробуйте другую степень,
  focal loss или пересэмплирование.
* **Архитектура.** Двунаправленная LSTM, GRU, одномерная свёртка, attention над окном.
  Модель крошечная — экспериментировать дёшево.
* **Постобработка.** Одиночный интервал класса 1 среди нулей, скорее всего, ложное
  срабатывание. Сглаживание предсказаний или требование минимальной длины события
  нередко даёт больше, чем смена архитектуры.
* **Порог решения.** `argmax` — не догма. Возьмите вероятность класса 1 и подберите порог
  под нужный баланс precision/recall.
* **Другой размер интервала.** Файл `fuel_raw.csv.gz` содержит исходные сообщения.
  5 минут — это наш выбор, а не закон природы.

### Что сдавать

Ноутбук, в котором:

1. воспроизводится бейзлайн (можно ссылкой на этот),
2. реализовано ваше улучшение,
3. в конце вызван `report(...)` — сравнение с бейзлайном по обеим метрикам,
4. есть короткий разбор: что сработало, что нет и почему вы так думаете.

### Отдельный вопрос со звёздочкой

В том же проекте есть вторая модель — для заправок с выключенным двигателем.
На тесте она даёт **accuracy 100% и F1 = 1.0** по целевому классу.

Её задача устроена так. Когда двигатель заглушен, датчик молчит, и в данных
остаётся провал. Всего таких провалов **11 758**, из них эксперт отметил заправками
**711**. Модель должна отличить одни от других по уровню до провала и после.

Возьмите разницу «уровень после минус уровень до» и поставьте на неё порог.
У заправок эта разница в среднем 36 литров, у обычных стоянок — ноль.

Вопросы:

1. Какой F1 даёт простой порог на этой разнице? (у нас вышло 0.968 при пороге 15 л)
2. Сколько тогда добавляет нейросеть поверх одного вычитания?
3. Что означает её «стопроцентная точность» и что нужно поменять, чтобы
   число стало осмысленным?


---
# 9. Решение: поднимаем событийный F1

**Протокол.** Разбиение `split` не трогаем. Внутри train те же 36 машин на обучение и 9 на валидацию,
что и в бейзлайне. Все решения (эпоха, признаки, порог) принимаются **только по валидации**;
тест считается для каждого варианта один раз и в выборе не участвует.

Сеть маленькая, и результат заметно зависит от случайной инициализации: разница в одну-две
заправки из 124 — это шум. Поэтому каждый вариант обучается с **тремя seed** и в таблице
стоит среднее ± разброс.

Варианты добавляются по одному поверх предыдущего:

| | изменение | гипотеза |
|---|---|---|
| A | бейзлайн | — |
| B | эпоха выбирается по **событийному** F1 на валидации, а не по точечному | мы отбираем модель по одной метрике, а судим по другой |
| C | + стандартизация признаков (статистики только по train) | уровень 0–100 л и разности около нуля в одном масштабе мешают LSTM |
| D | + нули датчика заменяются последним известным уровнем, флаг «датчик молчал» | переход 0 → 50 л сейчас выглядит как заправка на 50 л |
| E | + `ign`, `speed`, `n_msg`, «стоит с включённым зажиганием», время суток | заправка — это малая скорость при включённом зажигании |
| F | + двунаправленная LSTM, выход из центра окна | метка — центр окна, контекст справа должен помогать |
| G | + окно 11 вместо 7 | больше контекста для медленных заправок |

In [ ]:
# Общий конвейер для экспериментов: признаки -> окна -> обучение -> метрики.
import copy

SEEDS = [42, 1, 2]           # три запуска на вариант
EPOCHS_EXP = 30


def add_features_cfg(g, cfg):
    g = g.sort_values("ts").copy()
    base = g["fuel"]
    if cfg.get("ffill"):
        # 0 означает «датчик молчал»; для разностей берём последний известный уровень
        base = g["fuel"].replace(0.0, np.nan).ffill().fillna(0.0)
        g["fuel_ff"] = base
        g["silent"] = (g["fuel"] == 0).astype(float)
    for k in (1, 2, 3, 4):
        g[f"d{k}"] = base.diff(k).fillna(0.0)
    hour = g["ts"].dt.hour + g["ts"].dt.minute / 60
    g["hsin"] = np.sin(2 * np.pi * hour / 24)
    g["hcos"] = np.cos(2 * np.pi * hour / 24)
    g["slow_ign"] = ((g["speed"] < 3) & (g["ign"] > 0.5)).astype(float)
    return g


def features_cfg(cfg):
    f = ["fuel_ff" if cfg.get("ffill") else "fuel", "d1", "d2", "d3", "d4"]
    if cfg.get("ffill"):
        f.append("silent")
    if cfg.get("extra"):
        f += ["ign", "speed", "n_msg", "slow_ign", "hsin", "hcos"]
    return f


def windows_cfg(frame, feats, L):
    X, y, veh, ts = [], [], [], []
    c = L // 2
    for v, g in frame.groupby("vehicle"):
        vals = g[feats].values.astype("float32")
        lab = g["label"].values.astype("int64")
        times = g["ts"].values
        for i in range(len(g) - L + 1):
            X.append(vals[i:i + L]); y.append(lab[i + c]); veh.append(v); ts.append(times[i + c])
    return np.asarray(X), np.asarray(y), pd.DataFrame({"vehicle": veh, "ts": pd.to_datetime(ts)})


class Net(nn.Module):
    def __init__(self, n_features, cfg):
        super().__init__()
        self.center = cfg.get("head") == "center"
        bi = cfg.get("bi", False)
        self.rnn = nn.LSTM(n_features, 64, 2, batch_first=True, dropout=0.2, bidirectional=bi)
        self.fc = nn.Linear(64 * (2 if bi else 1), 3)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, x.shape[1] // 2 if self.center else -1, :])


raw = pd.read_csv(LOCAL, parse_dates=["ts"])          # исходные данные без признаков бейзлайна
_cache = {}


def prepare(cfg):
    key = (cfg.get("ffill", False), cfg.get("extra", False), cfg.get("L", 7), cfg.get("norm", False))
    if key in _cache:
        return _cache[key]
    d = pd.concat([add_features_cfg(g, cfg) for _, g in raw.groupby("vehicle")])
    feats, L = features_cfg(cfg), cfg.get("L", 7)
    parts = [windows_cfg(d[d.split.eq("train") & ~d.vehicle.isin(VAL_VEH)], feats, L),
             windows_cfg(d[d.vehicle.isin(VAL_VEH)], feats, L),
             windows_cfg(d[d.split == "test"], feats, L)]
    if cfg.get("norm"):
        flat = parts[0][0].reshape(-1, len(feats))
        mu, sd = flat.mean(0), flat.std(0) + 1e-6          # статистики только по train
        parts = [(((X - mu) / sd).astype("float32"), y, m) for X, y, m in parts]
    _cache[key] = (feats, parts)
    return _cache[key]


def proba(net, X):
    net.eval()
    with torch.no_grad():
        return torch.cat([torch.softmax(net(torch.from_numpy(X[i:i + 4096]).to(DEVICE)), 1).cpu()
                          for i in range(0, len(X), 4096)]).numpy()


def train_cfg(cfg, seed):
    if CACHED:
        assert CACHED['CONFIGS']==CONFIGS and CACHED['SEEDS']==SEEDS
        name=next(name for name,entry in CONFIGS.items() if entry==cfg)
        Pva,Pte=CACHED['probs_store'][(name,seed)]
        _,(_,(_,yva,_),(_,yte,_))=prepare(cfg)
        assert len(Pva)==len(yva) and len(Pte)==len(yte)
        return None,Pva,Pte
    torch.manual_seed(seed); np.random.seed(seed)
    feats, ((Xtr, ytr, _), (Xva, yva, mva), (Xte, yte, mte)) = prepare(cfg)
    net = Net(len(feats), cfg).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=class_weights(ytr).to(DEVICE))
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2)
    loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)),
                        batch_size=512, shuffle=True)
    best, state = -1.0, None
    for epoch in range(EPOCHS_EXP):
        net.train()
        for xb, yb in loader:
            opt.zero_grad()
            crit(net(xb.to(DEVICE)), yb.to(DEVICE)).backward()
            opt.step()
        pv = proba(net, Xva).argmax(1)
        score = (event_metrics(yva, pv, mva, verbose=False)["f1"] if cfg.get("select") == "event"
                 else f1_score(yva, pv, labels=[1], average="macro", zero_division=0))
        sch.step(score)
        if score > best:
            best, state = score, copy.deepcopy(net.state_dict())
    net.load_state_dict(state)
    folder=Path('fuel_artifacts/models');folder.mkdir(parents=True,exist_ok=True)
    name=next(name for name,entry in CONFIGS.items() if entry==cfg).split()[0]
    torch.save({'state_dict':{k:v.cpu() for k,v in net.state_dict().items()},'cfg':cfg,
                'features':feats,'seed':seed},folder/f'{name}_seed{seed}.pt')
    return net, proba(net, Xva), proba(net, Xte)


def point_f1(y, p):
    return f1_score(y, p, labels=[1], average="macro", zero_division=0)


print("устройство:", DEVICE, "| валидационные машины:", sorted(VAL_VEH))


In [ ]:
# Лестница экспериментов. На GPU занимает ~5 минут, на CPU ~25.
CONFIGS = {
    "A  бейзлайн":                         {},
    "B  эпоха по событийному F1":           {"select": "event"},
    "C  + нормализация":                    {"select": "event", "norm": True},
    "D  + уровень через нули датчика":      {"select": "event", "norm": True, "ffill": True},
    "E  + ign, speed, n_msg, время":        {"select": "event", "norm": True, "ffill": True, "extra": True},
    "F  + BiLSTM, выход из центра":         {"select": "event", "norm": True, "ffill": True, "extra": True,
                                             "bi": True, "head": "center"},
    "G  + окно 11":                         {"select": "event", "norm": True, "ffill": True, "extra": True,
                                             "bi": True, "head": "center", "L": 11},
}

rows, probs_store = [], {}
t0 = time.time()
for name, cfg in CONFIGS.items():
    _, (_, (_, yva, mva), (_, yte, mte)) = prepare(cfg)
    for seed in SEEDS:
        net, Pva, Pte = train_cfg(cfg, seed)
        probs_store[(name, seed)] = (Pva, Pte)
        pv, pt = Pva.argmax(1), Pte.argmax(1)
        ev_t = event_metrics(yte, pt, mte, verbose=False)
        rows.append({"вариант": name, "seed": seed,
                     "val event F1": event_metrics(yva, pv, mva, verbose=False)["f1"],
                     "test event F1": ev_t["f1"], "test point F1": point_f1(yte, pt),
                     "TP": ev_t["tp"], "FP": ev_t["fp"], "FN": ev_t["fn"]})
    print(f"{name:40s} готово, {time.time() - t0:5.0f} c")

res = pd.DataFrame(rows)
summary = (res.groupby("вариант", sort=False)[["val event F1", "test event F1", "test point F1", "FP", "FN"]]
              .agg(["mean", "std"]))
pretty = pd.DataFrame({
    col: summary[col]["mean"].map("{:.3f}".format) + " ± " + summary[col]["std"].map("{:.3f}".format)
    for col in ["val event F1", "test event F1", "test point F1"]})
pretty["FP (среднее)"] = summary["FP"]["mean"].round(1)
pretty["FN (среднее)"] = summary["FN"]["mean"].round(1)
pretty
from pathlib import Path
import pickle
OUT=Path('fuel_artifacts');OUT.mkdir(exist_ok=True)
keys=['baseline','y_test','y_pred','meta_test','probs_store','CONFIGS','SEEDS','VAL_VEH','raw','history','FEATURES','CENTER','X_test','X_val','y_val','meta_val']
with open(OUT/'training_checkpoint.pkl','wb') as f:pickle.dump({k:globals()[k] for k in keys if k in globals()},f,protocol=5)
torch.save({'state_dict':{k:v.cpu() for k,v in model.state_dict().items()},'features':FEATURES,'window':7},OUT/'baseline_weights.pt')
print('TRAINING_CHECKPOINT_SAVED')


In [ ]:
# Тот же результат на графике: валидация (по ней выбираем) и тест (только для отчёта)
m = res.groupby("вариант", sort=False)[["val event F1", "test event F1"]].agg(["mean", "std"])
x = np.arange(len(m))
fig, ax = plt.subplots(figsize=(9, 3.4))
for k, (col, color) in enumerate([("val event F1", "#9a9a9a"), ("test event F1", "#1a1a1a")]):
    ax.bar(x + (k - 0.5) * 0.38, m[col]["mean"], 0.38, yerr=m[col]["std"], capsize=3,
           color=color, label=col.replace(" event F1", ""))
ax.axhline(0.90, ls="--", lw=1, color="#c0392b")
ax.text(len(m) - 0.5, 0.905, "планка «хорошо» 0.90", ha="right", va="bottom", fontsize=9, color="#c0392b")
ax.set_xticks(x, [n.split()[0] for n in m.index])
ax.set_ylim(0.6, 1.02)
ax.set_ylabel("событийный F1")
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 9.1. Порог по вероятности: где рост становится нечестным

Вместо `argmax` можно объявлять заправкой всё, где вероятность класса 1 выше порога,
и подобрать порог на валидации. Посмотрим, что при этом происходит с **обеими** метриками
и со средней длиной предсказанного события.

In [ ]:
def decide(P, thr):
    if thr is None:
        return P.argmax(1)
    return np.where(P[:, 1] >= thr, 1, np.where(P[:, 2] > P[:, 0], 2, 0))


def mean_block_len(y, meta, cls=1):
    lens = []
    for _, sub in meta.groupby("vehicle"):
        m = np.pad(y[sub.index.to_numpy()] == cls, (1, 1)).astype(int)
        d = np.diff(m)
        lens += list(np.where(d == -1)[0] - np.where(d == 1)[0])
    return np.mean(lens) if lens else 0.0


DEMO = "C  + нормализация"            # для примера берём вариант C, seed 42
_, (_, (_, yva, mva), (_, yte, mte)) = prepare(CONFIGS[DEMO])
Pva, Pte = probs_store[(DEMO, 42)]

thr_rows = []
for thr in [0.03, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, None]:
    pv, pt = decide(Pva, thr), decide(Pte, thr)
    thr_rows.append({"порог": "argmax" if thr is None else thr,
                     "val event F1": event_metrics(yva, pv, mva, verbose=False)["f1"],
                     "val point F1": point_f1(yva, pv),
                     "test event F1": event_metrics(yte, pt, mte, verbose=False)["f1"],
                     "test point F1": point_f1(yte, pt),
                     "длина события, интервалов": mean_block_len(pt, mte)})
thr_tab = pd.DataFrame(thr_rows).set_index("порог")
print(f"средняя длина эталонной заправки в тесте: {mean_block_len(yte, mte):.2f} интервала")
thr_tab.round(3)


С низким порогом событийный F1 упирается в единицу, но точечный F1 **падает**, а предсказанные
события становятся заметно длиннее настоящих. Модель не стала лучше находить заправки — она
размазывает метку на соседние интервалы, и широкий блок «задевает» эталон с допуском ±5 минут.
Событийная метрика это не штрафует: блок, накрывший две заправки, засчитывается за обе.

Порог «в лоб» по максимуму событийного F1 мы **не берём**: прирост виден только на метрике,
у которой есть известная дыра. Честное правило — повышать событийный F1, **не теряя точечного**.
Так и подбираем ниже: порог принимается, только если на валидации событийный F1 вырос,
а точечный F1 не упал по сравнению с `argmax`.

In [ ]:
# «Безопасный» порог: подбирается на валидации, тест в выборе не участвует
THR_GRID = list(np.round(np.arange(0.05, 0.91, 0.05), 2))


def safe_threshold(Pva, yva, mva):
    p0 = Pva.argmax(1)
    base_point = point_f1(yva, p0)
    best_f, best_t = event_metrics(yva, p0, mva, verbose=False)["f1"], None
    for t in THR_GRID:
        pv = decide(Pva, t)
        f = event_metrics(yva, pv, mva, verbose=False)["f1"]
        if f > best_f + 1e-9 and point_f1(yva, pv) >= base_point:
            best_f, best_t = f, t
    return best_t


rows2 = []
for name, cfg in CONFIGS.items():
    _, (_, (_, yva, mva), (_, yte, mte)) = prepare(cfg)
    for seed in SEEDS:
        Pva, Pte = probs_store[(name, seed)]
        for mode in ("argmax", "безопасный порог"):
            t = None if mode == "argmax" else safe_threshold(Pva, yva, mva)
            pv, pt = decide(Pva, t), decide(Pte, t)
            rows2.append({"вариант": name, "решение": mode, "seed": seed, "порог": t,
                          "val event F1": event_metrics(yva, pv, mva, verbose=False)["f1"],
                          "val point F1": point_f1(yva, pv),
                          "test event F1": event_metrics(yte, pt, mte, verbose=False)["f1"],
                          "test point F1": point_f1(yte, pt)})
res2 = pd.DataFrame(rows2)
cols = ["val event F1", "val point F1", "test event F1", "test point F1"]
agg = res2.groupby(["вариант", "решение"], sort=False)[cols].mean()

# итоговый вариант выбирается автоматически: лучший средний событийный F1 на валидации
FINAL, FINAL_MODE = agg["val event F1"].idxmax()
print(f"лучшее по валидации: {FINAL.split()[0]} / {FINAL_MODE}")
agg.round(3)


## 9.2. Итоговая модель

Итоговый вариант выбран ячейкой выше **только по валидации** — лучший средний событийный F1
по трём seed среди всех вариантов и обоих способов принятия решения. Ниже — тот же
`report(...)`, что и для бейзлайна, для этого варианта на seed 42.

In [ ]:
feats_f,(_,(_,yva_f,mva_f),(_,yte_f,mte_f))=prepare(CONFIGS[FINAL])
Pva_f,Pte_f=probs_store[(FINAL,42)]
thr_f=None if FINAL_MODE=='argmax' else safe_threshold(Pva_f,yva_f,mva_f)
y_pred_final=decide(Pte_f,thr_f)
base_frame=meta_test.assign(truth=y_test,base_prediction=y_pred)
final_frame=mte_f.assign(final_truth=yte_f,final_prediction=y_pred_final)
matched=base_frame.merge(final_frame,on=['vehicle','ts'],how='inner',validate='one_to_one')
assert len(matched)==len(final_frame)
assert (matched.truth==matched.final_truth).all()
y_test=matched.truth.to_numpy(dtype='int64')
y_pred=matched.base_prediction.to_numpy(dtype='int64')
y_pred_final=matched.final_prediction.to_numpy(dtype='int64')
meta_test=matched[['vehicle','ts']].reset_index(drop=True)
print('Итог:',FINAL,'/',FINAL_MODE,'порог',thr_f,'общих тестовых окон',len(matched))
baseline=report(y_test,y_pred,meta_test,'Baseline на общем наборе временных точек')
final=report(y_test,y_pred_final,meta_test,'Итог на том же наборе временных точек')


In [ ]:
# Какие заправки модель всё ещё пропускает и где ложные срабатывания
def event_errors(y_true, y_pred, meta):
    out = []
    for v, sub in meta.groupby("vehicle"):
        pos = sub.index.to_numpy(); times = sub["ts"].values
        t_true, t_pred = blocks(y_true[pos] == 1, times), blocks(y_pred[pos] == 1, times)
        t_amb = blocks(y_true[pos] == 2, times)
        near = lambda a, b, lst: any(a <= te_ + TOLERANCE and b >= ts_ - TOLERANCE for ts_, te_ in lst)
        out += [(v, "пропуск", s) for s, e in t_true if not near(s, e, t_pred)]
        out += [(v, "ложное", s) for s, e in t_pred if not near(s, e, t_true) and not near(s, e, t_amb)]
    return out


errs = event_errors(y_test, y_pred_final, meta_test)
print(f"ошибок итоговой модели на тесте: {len(errs)}")
for v, kind, t in errs:
    print(f"  {v}  {kind:8s} {pd.Timestamp(t):%d.%m.%Y %H:%M}")
for v, kind, t in errs[:3]:
    t = pd.Timestamp(t)
    w = df[(df.vehicle == v) & (df.ts.between(t - pd.Timedelta("30min"), t + pd.Timedelta("30min")))]
    print(f"\n{v} {kind} {t:%d.%m %H:%M}")
    print(w[["ts", "fuel", "ign", "speed", "n_msg", "label"]].to_string(index=False))


---
# 10. Вопрос со звёздочкой: «100% точности» у модели для выключенного двигателя

Данных второй модели в ноутбуке нет, но их можно восстановить из сырых сообщений
`fuel_raw.csv.gz`. Провал — это серия сообщений с `fuel_raw = 0` (датчик молчит),
заправка — провал, рядом с которым эксперт поставил отметку `marked`. Короткие провалы
(меньше 2 минут) — это не стоянка с заглушенным двигателем, а единичные выпадения
сообщений во время движения, их отбрасываем.

In [ ]:
RAW_URL = "https://raw.githubusercontent.com/hehehelp123/fuel-refuel-lab/main/data/fuel_raw.csv.gz"
if not os.path.exists("fuel_raw.csv.gz"):
    print("качаю сырые сообщения (29 МБ)...")
    urllib.request.urlretrieve(RAW_URL, "fuel_raw.csv.gz")

r = pd.read_csv("fuel_raw.csv.gz", usecols=["vehicle", "ts", "fuel_raw", "fuel_l", "marked"],
                parse_dates=["ts"]).sort_values(["vehicle", "ts"], kind="stable")
gap_rows = []
for v, g in r.groupby("vehicle", sort=False):
    silent = g.fuel_raw.values == 0
    fl, mk, ts = g.fuel_l.values, g.marked.values, g.ts.values
    d = np.diff(np.r_[0, silent.astype(int), 0])
    for s, e in zip(np.where(d == 1)[0], np.where(d == -1)[0]):     # провал = сообщения [s, e)
        if s == 0 or e >= len(g):
            continue
        gap_rows.append({"vehicle": v, "dur_min": (ts[e] - ts[s - 1]) / np.timedelta64(1, "m"),
                         "before": fl[s - 1], "after": fl[e], "diff": fl[e] - fl[s - 1],
                         "is_fill": int(mk[s - 1:e + 1].any())})
gaps = pd.DataFrame(gap_rows)
gaps = gaps[gaps.dur_min >= 2].reset_index(drop=True)
gaps["split"] = gaps.vehicle.map(df.drop_duplicates("vehicle").set_index("vehicle").split)
gaps.loc[gaps.vehicle.isin(VAL_VEH), "split"] = "val"
del r

print(f"провалов: {len(gaps)}, из них заправок: {gaps.is_fill.sum()} "
      f"(в условии задачи: 11 758 и 711)")
print(gaps.groupby("is_fill")["diff"].agg(["count", "mean", "median"]).round(1)
          .rename(index={0: "стоянка", 1: "заправка"}))


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

print(f"{'порог, л':>9} | {'precision':>9} | {'recall':>6} | {'F1':>5}")
print("-" * 40)
val_gaps = gaps[gaps.split == "val"]
if val_gaps.empty:
    raise ValueError("Нет валидационных событий для подбора порога")
test_gaps = gaps[gaps.split == "test"]
if test_gaps.empty:
    raise ValueError("Нет тестовых событий")
thr_f1 = {}
for thr in [1, 3, 5, 8, 10, 12, 15, 20, 25]:
    pred_val = (val_gaps["diff"] > thr).astype(int)
    thr_f1[thr] = f1_score(val_gaps.is_fill, pred_val, zero_division=0)
    print(f"{thr:>9} | {precision_score(val_gaps.is_fill, pred_val, zero_division=0):>9.3f} | "
          f"{recall_score(val_gaps.is_fill, pred_val, zero_division=0):>6.3f} | {thr_f1[thr]:>5.3f}")
best_thr = max(thr_f1, key=thr_f1.get)
pred_test = (test_gaps["diff"] > best_thr).astype(int)
print(f"\nПорог, выбранный на val: {best_thr} л")
print(f"Тест: F1 = {f1_score(test_gaps.is_fill, pred_test, zero_division=0):.3f}; "
      f"precision = {precision_score(test_gaps.is_fill, pred_test, zero_division=0):.3f}; "
      f"recall = {recall_score(test_gaps.is_fill, pred_test, zero_division=0):.3f}")
print(f"Тест при фиксированном пороге 15 л: "
      f"F1 = {f1_score(test_gaps.is_fill, (test_gaps['diff'] > 15).astype(int), zero_division=0):.3f}")

te = gaps[gaps.split == "test"]
print(f"\nтестовые машины: {len(te)} провалов, из них заправок всего {te.is_fill.sum()}")
print(f"accuracy константы «никогда не заправка» на тесте: "
      f"{accuracy_score(te.is_fill, np.zeros(len(te))):.3f}")
n = int(te.is_fill.sum())
print(f"если модель нашла {n} из {n}: 95% доверительный интервал для recall "
      f"[{0.025 ** (1 / n):.2f}; 1.00] (Клоппер — Пирсон)")

fig, ax = plt.subplots(figsize=(8, 3))
bins = np.linspace(-20, 60, 81)
ax.hist(gaps[gaps.is_fill == 0]["diff"].clip(-20, 60), bins=bins, color="#9a9a9a", label="стоянка")
ax.hist(gaps[gaps.is_fill == 1]["diff"].clip(-20, 60), bins=bins, color="#1a1a1a", alpha=0.85, label="заправка")
ax.axvline(best_thr, ls="--", color="#c0392b", lw=1.2)
ax.set_yscale("log")
ax.set_xlabel("уровень после провала минус уровень до, литров")
ax.set_ylabel("число провалов")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Ответы по дополнительному пороговому методу

Порог для заправки при выключенном двигателе также выбирается на validation; точное качество на test напечатано предыдущей ячейкой. Этот эксперимент относится к классу 2 и не подменяет основную задачу класса 1. Нельзя сравнивать полученные значения без указания целевого класса, правил выделения провалов и допустимых временных отклонений.

In [ ]:
from pathlib import Path
import json
OUT=Path('fuel_artifacts');OUT.mkdir(exist_ok=True)
res.to_csv(OUT/'experiment_ladder.csv',index=False)
res2.to_csv(OUT/'decision_thresholds.csv',index=False)
threshold_validation.to_csv(OUT/'baseline_rule_validation.csv',index=False)
pd.DataFrame(errs,columns=['vehicle','kind','time']).to_csv(OUT/'final_errors.csv',index=False)
meta_test.assign(truth=y_test,baseline=y_pred,final=y_pred_final).to_csv(OUT/'final_predictions.csv',index=False)
summary={'selected':FINAL,'decision':FINAL_MODE,'probability_threshold':thr_f,'test_windows':len(y_test),
 'baseline_common':baseline,'final_common':final,'simple_rule_threshold':baseline_rule_threshold,
 'rule_test':frozen_threshold_test,'training_cache_reused':bool(CACHED),
 'note':'Порог выбран только на validation; итог сравнивается на общих временных точках.'}
(OUT/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,default=float))
import shutil
selected_file=OUT/'models'/f'{FINAL.split()[0]}_seed42.pt'
if selected_file.exists():shutil.copy2(selected_file,OUT/'selected_model.pt')
print('FRESH_FUEL_EXPORT',summary)


## Как читать результат

Сравнение выполняется на одном наборе временных точек тестовых машин. Конфигурация и порог выбираются на validation; проверяются событийный и точечный F1. В репозитории отсутствуют старые pickle-кэши, повторное обучение включено по умолчанию.

Рост событийного F1 сопровождается снижением точечного F1. Цифры относятся к конкретным данным и протоколу. BiLSTM с центральным окном использует будущие точки и подходит для офлайн-анализа или работы с задержкой.

Сохранённая история исходного эксперимента находится отдельно в `results/recorded_metrics.json`. Полное GPU-обучение при подготовке этой версии не выполнялось.